# EcoPredict — Validaciones adicionales

**Experimento 1.** Reconstrucción de la cantidad anual de residuos domiciliarios mediante la fórmula administrativa del MINAM.

**Experimento 2.** Evaluación de XGBoost en 2024 utilizando variables históricas, con comparación frente a persistencia y tendencia lineal.

**Datos de entrada:** `dataset3_proporcion.csv`. La evaluación de 2024 no utiliza población ni GPC de ese año, aunque requiere confirmar la disponibilidad temporal de los registros de 2023.

Dependencias: `pandas`, `numpy`, `scikit-learn` y `xgboost`. Coloca `dataset3_proporcion.csv` junto al notebook o ajusta su ubicación en la celda de ejecución.

In [ ]:
import pandas as pd, numpy as np, json, hashlib, platform
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
ROOT=Path.cwd(); OUT=ROOT/'EcoPredict_validacion_adicional'; OUT.mkdir(exist_ok=True)
possible=[ROOT/'dataset3_proporcion(1).csv',ROOT/'dataset3_proporcion.csv',Path('/content/drive/MyDrive/Colab Notebooks/dataset3_proporcion.csv'),Path('/content/drive/MyDrive/Colab Notebooks/dataset3_proporcion(1).csv')]
path=next((p for p in possible if p.exists()),None)
if path is None: raise FileNotFoundError('Sube dataset3_proporcion.csv a Colab o conecta Google Drive y ajusta la ruta.')
raw=pd.read_csv(path,sep=';',encoding='latin1',low_memory=False); raw.columns=raw.columns.str.strip().str.upper()
d=raw[raw.PROVINCIA.astype(str).str.strip().str.upper().eq('LIMA')].copy(); d=d[d.PERIODO.between(2019,2024)].copy();d['DISTRITO']=d.DISTRITO.str.strip().str.upper().replace({'MAGDALENA VIEJA':'PUEBLO LIBRE'});d=d.sort_values(['DISTRITO','PERIODO']).reset_index(drop=True)
assert len(d)==258 and d.DISTRITO.nunique()==43
cols=[c for c in d if c.startswith('QRESIDUOS_') and c!='QRESIDUOS_DOM']; assert len(cols)==33
for c in cols: d[c]=d[c].fillna(0)
d['FORMULA_MINAM']=d.GPC_DOM*d.POB_URBANA*365/1000
for col in ['QRESIDUOS_DOM','FORMULA_MINAM']: d[col]=pd.to_numeric(d[col],errors='raise')
def metric(y,p):
 y=np.array(y,dtype=float);p=np.array(p,dtype=float)
 return {'n':len(y),'R2':float(r2_score(y,p)),'RMSE_t_anio':float(np.sqrt(mean_squared_error(y,p))),'MAE_t_anio':float(mean_absolute_error(y,p)),'MAPE_pct':float(100*np.mean(np.abs(y-p)/np.abs(y))),'MaxAE_t_anio':float(np.max(np.abs(y-p)))}
formula_all=metric(d.QRESIDUOS_DOM,d.FORMULA_MINAM)
formula_test=metric(d.loc[d.PERIODO>=2023,'QRESIDUOS_DOM'],d.loc[d.PERIODO>=2023,'FORMULA_MINAM'])
formula_2024=metric(d.loc[d.PERIODO==2024,'QRESIDUOS_DOM'],d.loc[d.PERIODO==2024,'FORMULA_MINAM'])
# Variables construidas a partir de años anteriores
for lag in (1,2):d[f'LAG{lag}_TOTAL']=d.groupby('DISTRITO').QRESIDUOS_DOM.shift(lag)
d['CRECIMIENTO_TOTAL']=d.LAG1_TOTAL-d.LAG2_TOTAL
d['ROLL_MEAN_2_TOTAL']=d.groupby('DISTRITO').QRESIDUOS_DOM.transform(lambda x:x.shift(1).rolling(2,min_periods=1).mean())
for c in cols:
 d[c+'_PROP_LAG1']=d.groupby('DISTRITO')[c].shift(1)/(d.LAG1_TOTAL+1e-6)
d=d.dropna(subset=['LAG1_TOTAL','LAG2_TOTAL','CRECIMIENTO_TOTAL','ROLL_MEAN_2_TOTAL']+[c+'_PROP_LAG1' for c in cols]).copy()
assert len(d)==172 and len(d[d.PERIODO==2024])==43
train=d.PERIODO<=2023; test=d.PERIODO==2024
hist=['LAG1_TOTAL','LAG2_TOTAL','CRECIMIENTO_TOTAL','ROLL_MEAN_2_TOTAL']
options={'XGBoost_historial': ['DISTRITO','PERIODO']+hist,'XGBoost_historial_composicion':['DISTRITO','PERIODO']+hist+[c+'_PROP_LAG1' for c in cols]}
rows=[]; preds={}
y=d.loc[test,'QRESIDUOS_DOM'].to_numpy()
for name, features in options.items():
 preprocess=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['DISTRITO']),('num','passthrough',[c for c in features if c!='DISTRITO'])]); model=Pipeline([('prep',preprocess),('model',XGBRegressor(n_estimators=500,learning_rate=0.05,max_depth=5,subsample=.9,colsample_bytree=.9,reg_alpha=.2,reg_lambda=1.5,random_state=42,n_jobs=1,objective='reg:squarederror'))]);model.fit(d.loc[train,features],d.loc[train,'QRESIDUOS_DOM']);pr=model.predict(d.loc[test,features]);preds[name]=pr;rows.append({'Modelo':name,**metric(y,pr)})
for name,pr in [('Persistencia_2023',d.loc[test,'LAG1_TOTAL'].to_numpy()),('Tendencia_lineal_2022_2023',(2*d.loc[test,'LAG1_TOTAL']-d.loc[test,'LAG2_TOTAL']).to_numpy())]:preds[name]=pr;rows.append({'Modelo':name,**metric(y,pr)})
# Referencia administrativa con datos del mismo año; no representa un pronóstico
rows.append({'Modelo':'Formula_MINAM_con_datos_2024_NO_prospectivo',**formula_2024})
metrics=pd.DataFrame(rows).sort_values('RMSE_t_anio');metrics.to_csv(OUT/'metricas_2024.csv',index=False)
details=d.loc[test,['DISTRITO','PERIODO','QRESIDUOS_DOM','LAG1_TOTAL','LAG2_TOTAL']].copy(); details['Formula_MINAM_mismo_anio']=d.loc[test,'FORMULA_MINAM'];
for name,pr in preds.items():details[name]=pr
details.to_csv(OUT/'predicciones_2024.csv',index=False)
meta={'csv_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'original_rows':len(raw),'lima_rows':258,'modelable_rows':len(d),'training_rows':int(train.sum()),'test_rows':int(test.sum()),'num_composition_features':len(cols),'formula_all':formula_all,'formula_test_2023_2024':formula_test,'formula_2024':formula_2024,'versions':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'xgboost':__import__('xgboost').__version__,'scikit_learn':__import__('sklearn').__version__},'caveat':'All 2024 models avoid 2024 GPC/population. Still require observed 2023 waste and composition for training-time inputs; actual publication time must be verified.'}
(OUT/'resumen.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2))
print('formula all',formula_all);print('formula 2024',formula_2024);print(metrics.to_string(index=False));print('saved',OUT)


## Consideraciones de interpretación

- La fórmula del MINAM utiliza GPC y población urbana del mismo año. Su evaluación corresponde a una reconstrucción del valor reportado.
- La evaluación histórica de 2024 excluye población y GPC de ese año, pero depende de observaciones de 2023 cuya fecha de publicación debe comprobarse.
- XGBoost histórico presenta un RMSE ligeramente menor que persistencia, mientras que persistencia obtiene menor MAE en esta partición.
- Las discrepancias observadas en Rímac (2021–2022) se conservan para su verificación con la fuente original.